# 02 · Lexicons

Chinese MentalBERT masks words from a **depression lexicon** (Li et al., 2020) first during
pre-training. That lexicon is public and is all the baselines need.

The **multi-disorder lexicon** (520 words, `docs/LEXICON.md`) is an optional variant: candidate
words mined from MATCHA training users, kept only if two LLM annotators both judge them
mental-health related. It feeds `configs/pipeline_multidisorder.yaml` only, and none of the main
results.

| | |
|---|---|
| Hardware | CPU |
| Time | ~1 min (depression lexicon); the optional part ~10 min plus API calls |
| Needs | internet; for the optional part a DeepSeek API key in `.env` (~$0.26) and annotator A's word list |
| Writes | `lexicons/` (never committed) |

> **Data use.** MATCHA is under a Data Use Agreement. These notebooks print counts and scores only, never posts; keep it that way if you add cells, and do not commit notebooks with outputs. Only candidate *words* (with the disorder's name) are ever sent to an API, never posts.

In [ ]:
import sys
from pathlib import Path

# make `matcha_bench` importable from a fresh clone (not needed after `pip install -e .`)
sys.path.insert(0, str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"))
from matcha_bench.notebook import check, setup, step

ROOT = setup()  # every step runs from the repository root
print(ROOT)

## 1. The depression lexicon (the Chinese MentalBERT recipe)

In [ ]:
step("scripts/build_depression_lexicon.py")

from matcha_bench.lexicon.io import load_lexicon

check("depression lexicon words", len(load_lexicon("lexicons/lexicon_depression.txt")), 1_470)

## 2. Optional: the multi-disorder lexicon

1. **Mining** (local): per disorder vs clean controls, weighted log-odds with an informative
   Dirichlet prior; z ≥ 3, used by ≥ 5 distinct users, top 1,500 per disorder.
2. **Annotator A**: one pass of an LLM over the 4,269 unique candidate words, with the criteria in
   `lexicons/README.md`. Its output is `lexicons/llm_annotations.csv` (`word,category`, one row per
   kept word). Ours is derived from DUA data and not distributed; any annotator (LLM or human) can
   produce a file in that format.
3. **Annotator B**: DeepSeek (`deepseek-chat`), each (disorder, word) judged 3 times at
   temperature 0.7, majority vote.
4. **Consensus**: a word is kept only if both annotators keep it.

In [ ]:
BUILD_MULTIDISORDER = False  # set to True to run this part

In [ ]:
if BUILD_MULTIDISORDER:
    step("scripts/mine_lexicon.py")
    import pandas as pd

    candidates = pd.read_csv("lexicons/candidates.csv")
    check("(disorder, word) candidates", len(candidates), 9_307)
    check("unique candidate words", candidates["word"].nunique(), 4_269)

In [ ]:
if BUILD_MULTIDISORDER:
    assert Path("lexicons/llm_annotations.csv").exists(), "annotator A's word list is needed (see above)"
    step("scripts/filter_lexicon_deepseek.py", "--dry-run")  # cost estimate, no API calls
    step("scripts/filter_lexicon_deepseek.py")               # annotator B
    step("scripts/compare_lexicon_annotations.py")
    import json

    agreement = json.loads(Path("lexicons/annotation_agreement.json").read_text(encoding="utf-8"))
    check("Cohen's kappa, words", round(agreement["word_level"]["cohen_kappa"], 2), 0.43)
    check("Cohen's kappa, (disorder, word) pairs", round(agreement["pair_level"]["cohen_kappa"], 2), 0.55)
    step("scripts/build_multidisorder_lexicon.py")
    check("multi-disorder lexicon words", len(load_lexicon("lexicons/lexicon_multidisorder.txt")), 520)

With other annotators the numbers will differ: ours were made in October 2026. Next:
`03_chinese_mentalbert.ipynb`.